In [ ]:
from GPU_Enabled_Combined_MT import *

In [ ]:
import os
import time
import copy
import pickle
import torch
import numpy as np
import pandas as pd
import optuna

# Assuming definitions like PolicyNet, MovieLensEnv, train_policy_gradient_batched,
# evaluate_policy, eval_all_ks, and data variables are imported or reside in this namespace.
# (Paste your data-loading, feature engineering, and core training functions here)

# ---------------------------------------------------------------------------
# Global Settings
# ---------------------------------------------------------------------------
RETAIN_DROP_LIMIT_PP = 2.0
TOP_SELECTION_K = 10
BASE_CSV_OUTPUT = "optuna_base_model_search.csv"

# ---------------------------------------------------------------------------
# Optuna Callback for Live CSV Tracking (Bypasses SQLite)
# ---------------------------------------------------------------------------
def log_study_to_csv(csv_path):
    def callback(study, trial):
        df = study.trials_dataframe()
        # Clean up column names for readability
        df.columns = df.columns.str.replace("user_attrs_", "")
        df.to_csv(csv_path, index=False)
    return callback

# ===========================================================================
# PHASE 1: Base Model Tuning (25 Trials)
# ===========================================================================
def objective_phase1(trial):
    # Suggest upstream hyperparameters
    t_lr = trial.suggest_float("train_lr", 1e-5, 1e-3, log=True)
    gamma = trial.suggest_float("gamma", 0.97, 0.99)
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    train_bs = trial.suggest_categorical("train_batch", [1, 2, 4])
    
    # Track metadata in study dataframe
    trial.set_user_attr("train_lr", t_lr)
    trial.set_user_attr("gamma", gamma)
    trial.set_user_attr("hidden_dim", hidden_dim)
    trial.set_user_attr("train_batch", train_bs)

    # Initialize and train from scratch
    env_tr = MovieLensEnv(trajectories_all, build_state_fn, candidate_movies)
    net = PolicyNet(state_dim, num_actions, hidden_dim=hidden_dim).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=t_lr)
    
    train_policy_gradient_batched(
        env_tr, net, opt,
        num_episodes=10_000,
        gamma=gamma,
        batch_size=train_bs,
    )
    
    net.eval()
    baseline = eval_all_ks(net, retain_trajectories, forget_trajectories, trajectories_all)
    h_c = baseline[TOP_SELECTION_K][4]  # base_combined_Hit
    n_c = baseline[TOP_SELECTION_K][5]  # base_combined_NDCG
    
    trial.set_user_attr("base_combined_Hit", h_c)
    trial.set_user_attr("base_combined_NDCG", n_c)
    
    # Save trial checkpoint model weights
    t_path = f"models/optuna_base_trial_{trial.number}.pt"
    torch.save(net.state_dict(), t_path)
    trial.set_user_attr("trained_model_path", t_path)
    
    return h_c  # Maximize combined Hit at K=10

def run_base_optimization():
    print("--- Starting Phase 1: Base Model Optimization ---")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_phase1, n_trials=25, callbacks=[log_study_to_csv(BASE_CSV_OUTPUT)])
    return study.best_trial.user_attrs

# ===========================================================================
# PHASE 2: Downstream Unlearning Constrained Tuning (25 Trials Per Method)
# ===========================================================================
def objective_unlearn(trial, method, base_config):
    # Load frozen base model values
    hidden_dim = int(base_config["hidden_dim"])
    t_model_path = base_config["trained_model_path"]
    
    # Suggest downstream unlearning hyperparameters
    u_lr = trial.suggest_float("unlearn_lr", 1e-5, 1e-3, log=True)
    u_iters = trial.suggest_int("unlearn_iters", 500, 2000, step=500)
    
    # Set default lambda values based on selected algorithm constraints
    if method in ["Ye_multi", "Ye_ApxI"]:
        lam = 1.0
    elif method == "Gradient_Ascent":
        lam = 0.0
    else:
        lam = trial.suggest_float("lambda_retain", 0.1, 2.0)
        
    trial.set_user_attr("unlearn_lr", u_lr)
    trial.set_user_attr("unlearn_iters", u_iters)
    trial.set_user_attr("lambda_retain", lam)

    # Initialize frozen policy state mapping
    net = PolicyNet(state_dim, num_actions, hidden_dim=hidden_dim).to(DEVICE)
    net.load_state_dict(torch.load(t_model_path, map_location=DEVICE))
    
    # Evaluate Baseline values right before running fine-tune execution loops
    baseline = eval_all_ks(net, retain_trajectories, forget_trajectories, trajectories_all)
    bh_r, bn_r, bh_f, bn_f = baseline[TOP_SELECTION_K][:4]

    # Execute respective unlearning function algorithms
    net_copy = copy.deepcopy(net)
    if method == "Ye_multi":
        unlearning_finetune_ye_multi(net_copy, forget_buffer=f_buf, retain_buffer=r_buf, num_iters=u_iters, lr=u_lr, lambda_retain=lam)
    elif method == "New_True_inf":
        unlearning_finetune_new_true_inf(net_copy, forget_buffer=f_buf, retain_buffer=r_buf, num_iters=u_iters, lr=u_lr, lambda_retain=lam)
    elif method == "Gradient_Ascent":
        env_for = MovieLensEnv(forget_trajectories, build_state_fn, candidate_movies)
        unlearning_gradient_ascent(env=env_for, policy_net=net_copy, num_iters=u_iters, lr=u_lr, batch_size=int(base_config["train_batch"]))

    # Evaluate Results Post-Unlearning
    after = eval_all_ks(net_copy, retain_trajectories, forget_trajectories, trajectories_all)
    h_r, n_r, h_f, n_f = after[TOP_SELECTION_K][:4]
    
    retain_drop_pp = (bh_r - h_r) * 100.0
    forget_drop_pp = (bh_f - h_f) * 100.0
    
    trial.set_user_attr("retain_drop_pp", retain_drop_pp)
    trial.set_user_attr("forget_drop_pp", forget_drop_pp)

    # Apply strict 2.0 pp penalty constraint
    if retain_drop_pp > RETAIN_DROP_LIMIT_PP:
        # Subtract from an out-of-bounds floor penalty base to force Optuna inside boundaries
        return -100.0 - (retain_drop_pp - RETAIN_DROP_LIMIT_PP)
        
    return forget_drop_pp  # Maximize forget drop

def run_unlearn_optimization(method, best_base_config):
    print(f"\n--- Starting Phase 2: Unlearning Optimization ({method}) ---")
    csv_filename = f"optuna_unlearn_search_{method}.csv"
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective_unlearn(trial, method, best_base_config), n_trials=25, callbacks=[log_study_to_csv(csv_filename)])
    return study.best_trial.user_attrs

In [ ]:
if __name__ == "__main__":
    os.makedirs("models", exist_ok=True)
    
    # 1. Optimize Base Model Performance (Approach B Global Control)
    best_base = run_base_optimization()
    print(f"🥇 Optimal Global Base Model Selected: Trial Match -> {best_base['trained_model_path']}")
    
    # Target methods benchmark suite
    target_methods = ["Ye_multi", "New_True_inf", "Gradient_Ascent"]
    best_unlearn_configs = {}
    
    # 2. Run Optuna sweeps across unlearning parameters
    for method in target_methods:
        best_unlearn_configs[method] = run_unlearn_optimization(method, best_base)
        
    # 3. Phase 3: 10-Seed Significance Evaluation Loop
    print("\n=========================================================")
    print("Starting Phase 3: 10-Seed Multi-Run Significance Phase")
    print("=========================================================")
    
    sig_results = []
    num_statistical_runs = 10
    
    for method in target_methods:
        config = best_unlearn_configs[method]
        
        # Skip evaluating parameters that failed the constraint limits completely across all 25 trials
        if config["retain_drop_pp"] > RETAIN_DROP_LIMIT_PP:
            print(f"⚠️ Warning: Method {method} failed to find a valid parameter profile satisfying constraints. Skipping.")
            continue
            
        print(f"🎲 Running 10-seed significance profile for: {method}")
        
        for seed_idx in range(1, num_statistical_runs + 1):
            # Isolate the exact unlearning initialization seed dynamically 
            set_seed(make_seed(config["unlearn_lr"], config["unlearn_iters"], method, "statistical_eval", seed_idx))
            
            # Load your frozen global base model weights
            net = PolicyNet(state_dim, num_actions, hidden_dim=int(best_base["hidden_dim"])).to(DEVICE)
            net.load_state_dict(torch.load(best_base["trained_model_path"], map_location=DEVICE))
            
            # Run fine-tune unlearn routines with explicit parameters extracted from the study CSVs
            net_copy = copy.deepcopy(net)
            if method == "Ye_multi":
                unlearning_finetune_ye_multi(net_copy, forget_buffer=f_buf, retain_buffer=r_buf, num_iters=int(config["unlearn_iters"]), lr=float(config["unlearn_lr"]), lambda_retain=float(config["lambda_retain"]))
            elif method == "New_True_inf":
                unlearning_finetune_new_true_inf(net_copy, forget_buffer=f_buf, retain_buffer=r_buf, num_iters=int(config["unlearn_iters"]), lr=float(config["unlearn_lr"]), lambda_retain=float(config["lambda_retain"]))
            elif method == "Gradient_Ascent":
                env_for = MovieLensEnv(forget_trajectories, build_state_fn, candidate_movies)
                unlearning_gradient_ascent(env=env_for, policy_net=net_copy, num_iters=int(config["unlearn_iters"]), lr=float(config["unlearn_lr"]), batch_size=int(best_base["train_batch"]))
                
            # Log metrics tracking distribution values
            after = eval_all_ks(net_copy, retain_trajectories, forget_trajectories, trajectories_all)
            h_r, n_r, h_f, n_f = after[TOP_SELECTION_K][:4]
            
            sig_results.append({
                "method": method,
                "run_idx": seed_idx,
                "unlearn_lr": config["unlearn_lr"],
                "unlearn_iters": config["unlearn_iters"],
                "lambda_retain": config["lambda_retain"],
                "retain_Hit": h_r,
                "retain_NDCG": n_r,
                "forget_Hit": h_f,
                "forget_NDCG": n_f
            })
            
    # Save the absolute final significance profiles directly to file
    df_sig = pd.DataFrame(sig_results)
    df_sig.to_csv("final_unlearning_significance_results.csv", index=False)
    print("\n✅ Execution loop complete. Statistical distribution dataset compiled to: final_unlearning_significance_results.csv")